#### **Biostars**

In [22]:
#!pip install requests beautifulsoup4

In [23]:
import re, time, json, csv, html
from pathlib import Path
from urllib.parse import urlencode
import requests
from bs4 import BeautifulSoup

In [24]:
# ------------------------
# Paths & output
# ------------------------
CONFIG_PATH = Path("../02-datasource/category.json")
OUT_DIR     = Path("../04-output/raw/biostars")
OUT_DIR.mkdir(parents=True, exist_ok=True)

JSONL_PATH  = OUT_DIR/"biostars_metagenomics.jsonl"
CSV_PATH    = OUT_DIR/"biostars_metagenomics.csv"
CACHE_IDS   = OUT_DIR/"seen_ids.txt"

HEADERS = {"User-Agent": "MetagenomicsCollector/1.1 (+research use; contact: mahouzonssou.akotenou@um6p.ma)"}

# Crawl config
PAGES_PER_QUERY = 5
SLEEP_BETWEEN_REQUESTS = 1.2

In [25]:
# ------------------------
# Load and compile config
# ------------------------
def load_category_config(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        cfg = json.load(f)
    return cfg

def compile_regex_list(patterns, flags=re.I):
    compiled = []
    for pat in patterns:
        try:
            compiled.append(re.compile(pat, flags))
        except re.error as e:
            try:
                compiled.append(re.compile(pat.encode("utf-8").decode("unicode_escape"), flags))
            except Exception:
                print(f"[warn] bad regex skipped: {pat} ({e})")
    return compiled


CFG = load_category_config(CONFIG_PATH)
GLOBAL_EXCLUDES = compile_regex_list(CFG.get("global_exclude_regex", []))
CATEGORY_RULES = []
for cat in CFG.get("categories", []):
    CATEGORY_RULES.append({
        "name": cat.get("name"),
        "tags": cat.get("tags", []),
        "includes": compile_regex_list(cat.get("include_regex_any", []))
    })

# Build discovery query terms:
# ------------------------
# - category names & tags
# - tool_aliases (keys and their variants)
# - a few helpful generic discover terms
QUERY_TERMS = set()

for cat in CFG.get("categories", []):
    if cat.get("name"):
        QUERY_TERMS.add(cat["name"])
    for t in cat.get("tags", []):
        QUERY_TERMS.add(t)

tool_aliases = CFG.get("tool_aliases", {})
for tool, variants in tool_aliases.items():
    QUERY_TERMS.add(tool)
    for v in variants:
        QUERY_TERMS.add(v)

QUERY_TERMS.update([
    "metagenome", "metagenomics", "microbiome shotgun",
    "metagenome assembly", "metagenome binning", "MAGs",
    "host removal", "contamination", "quality trimming",
    "alpha diversity", "beta diversity", "ANCOM", "DESeq2"
])

QUERY_TERMS = sorted({q for q in QUERY_TERMS if 2 <= len(q) <= 40})
print(f"Loaded {len(CATEGORY_RULES)} categories, "
      f"{len(GLOBAL_EXCLUDES)} global excludes, "
      f"{len(QUERY_TERMS)} discovery queries.")

Loaded 20 categories, 3 global excludes, 145 discovery queries.


In [35]:
from bs4 import BeautifulSoup
import re

def html_plain_text(html):
    try:
        return BeautifulSoup(html or "", "html.parser").get_text("\n", strip=True)
    except Exception:
        return (html or "").strip()

def extract_answer_ids_and_accept_flag(soup):
    """
    Return (answer_ids, accepted_id) from a Biostars thread HTML.
    """
    ans_ids = []
    accepted_id = None
    for block in soup.select('div.post.view.open[data-value]'):
        pid = block.get("data-value")
        if not pid:
            continue
        # Is this block an answer? (accepted or suggested)
        if block.select_one('div.body[itemprop="acceptedAnswer"], div.body[itemprop="suggestedAnswer"]'):
            ans_ids.append(pid)
            if block.select_one('div.body[itemprop="acceptedAnswer"]'):
                accepted_id = pid
    return ans_ids, accepted_id

def parse_thread_page(pid):
    """
    HYBRID: use HTML to enumerate answers (+ accepted), then API per post for content/metadata.
    Returns:
      {
        url, title, question_text, question_created_at,
        answers, best_answer, tags_html
      }
    """
    url = f"https://www.biostars.org/p/{pid}/"
    html_text = fetch(url)
    soup = BeautifulSoup(html_text, "html.parser")

    # Title from page (robust)
    title_el = soup.select_one('div.post.view .title[itemprop="name"]') or soup.select_one("div.title")
    title = title_el.get_text(strip=True) if title_el else ""

    # Question created time
    q_time_el = soup.select_one('div.post.view time[itemprop="dateCreated"]')
    q_created_at = q_time_el.get("datetime") if q_time_el and q_time_el.has_attr("datetime") else None

    # Tags on page (API also provides tag_val)
    tags = [a.get_text(strip=True) for a in soup.select("span.inplace-tags a.ptag")]

    # Use API to get clean question body & metadata too
    q_json = get_post_json(pid)
    q_text = html_plain_text(q_json.get("xhtml", ""))

    # Enumerate answers via HTML, then fetch each via API
    ans_ids, accepted_id = extract_answer_ids_and_accept_flag(soup)

    answers = []
    for aid in ans_ids:
        a_json = get_post_json(aid)
        a_text = html_plain_text(a_json.get("xhtml", ""))
        if len(a_text) < 20:
            continue
        answers.append({
            "text": a_text,
            "score": int(a_json.get("vote_count", 0) or 0),
            "accepted": (aid == accepted_id),
            "created_at": a_json.get("creation_date")
        })

    # Choose best: accepted if any, else highest score (tie-break by length)
    best = None
    if answers:
        acc = [a for a in answers if a["accepted"]]
        best = (sorted(acc, key=lambda x: (x["score"], len(x["text"])), reverse=True)[0]
                if acc else sorted(answers, key=lambda x: (x["score"], len(x["text"])), reverse=True)[0])

    return {
        "url": url,
        "title": title,
        "question_text": q_text,
        "question_created_at": q_json.get("creation_date") or q_created_at,
        "answers": answers,
        "best_answer": best,
        "tags_html": tags,
        # You can also return the API's tag_val here if you want:
        "tags_api": q_json.get("tag_val", "")
    }

def normalize_record(pid, thread_meta, post_json, search_query=""):
    q = (thread_meta.get("question_text") or "").strip()
    a_best = (thread_meta.get("best_answer", {}) or {}).get("text", "").strip()
    if not q or not a_best:
        return None

    # tags: prefer API tag_val, fallback to HTML
    tags_site = post_json.get("tag_val") or thread_meta.get("tags_html") or []
    if isinstance(tags_site, str):
        tags_site = [tags_site]

    full_text = " ".join([thread_meta.get("title",""), q, a_best, " ".join(tags_site)])
    if blocked_by_global_excludes(full_text):
        return None
    labels = apply_labels_from_config(full_text)

    return {
        "source": "biostars",
        "post_id": pid,
        "url": thread_meta["url"],
        "title": thread_meta.get("title", ""),
        "question": q,
        "question_created_at": thread_meta.get("question_created_at"),
        "answer": a_best,
        "answers_all": thread_meta.get("answers", []),  # <— full list with created_at/score/accepted
        "accepted": bool(thread_meta.get("best_answer", {}).get("accepted", False)),
        "score": thread_meta.get("best_answer", {}).get("score", 0),
        "tags_site": tags_site,
        "created_at": post_json.get("creation_date"),
        "has_accepted": post_json.get("has_accepted", False),
        "view_count": post_json.get("view_count", 0),
        "topic_labels": labels,
        "search_query": search_query
    }


In [36]:
# ------------------------
# Crawl (fixed: no premature breaks + simple diagnostics)
# ------------------------
seen = load_seen_ids()
collected = 0
drop_stats = {"empty_question": 0, "no_answer": 0, "excluded_or_other": 0}

with open(JSONL_PATH, "a", encoding="utf-8") as jf:
    for term in QUERY_TERMS:
        pages_scanned = 0
        print(f"\n=== TERM: {term} ===")
        for url in biostars_search_urls(term, pages=PAGES_PER_QUERY):
            try:
                html_text = fetch(url)
            except Exception as e:
                print("Search fetch failed:", url, e)
                sleep()
                continue
            sleep()
            pages_scanned += 1

            pids = extract_post_ids_from_search(html_text)
            if not pids:
                print(f"[{term}] no pids found on {url}")

            for pid in pids:
                if pid in seen:
                    continue
                try:
                    thread_meta = parse_thread_page(pid); sleep()
                    post_json   = get_post_json(pid);     sleep()
                    rec = normalize_record(pid, thread_meta, post_json, search_query=term)
                    if rec:
                        jf.write(json.dumps(rec, ensure_ascii=False) + "\n")
                        jf.flush()
                        collected += 1
                        seen.add(pid)
                        if collected % 25 == 0:
                            print(f"Collected {collected} items so far...")
                    else:
                        # basic drop diagnostics
                        if not (thread_meta.get("question_text") or "").strip():
                            drop_stats["empty_question"] += 1
                        elif not (thread_meta.get("best_answer") or {}).get("text"):
                            drop_stats["no_answer"] += 1
                        else:
                            drop_stats["excluded_or_other"] += 1
                except Exception as e:
                    print(f"[warn] failed pid={pid}: {e}")
                    sleep()

            save_seen_ids(seen)
            sleep()

        print(f"[{term}] pages_scanned={pages_scanned}")

print(f"\nDone. Collected {collected} items → {JSONL_PATH}")
print("Drop stats:", drop_stats)


=== TERM: ANCOM ===


KeyboardInterrupt: 

In [28]:
# ------------------------
# Emit CSV snapshot
# ------------------------
field_order = [
    "source","post_id","url","title","question","answer","accepted",
    "score","tags_site","created_at","has_accepted","view_count","topic_labels"
]

rows = []
with open(JSONL_PATH, "r", encoding="utf-8") as jf:
    for line in jf:
        obj = json.loads(line)
        obj["topic_labels"] = ";".join(obj.get("topic_labels", []))
        rows.append({k: obj.get(k, "") for k in field_order})

with open(CSV_PATH, "w", encoding="utf-8", newline="") as cf:
    w = csv.DictWriter(cf, fieldnames=field_order)
    w.writeheader()
    for r in rows:
        w.writerow(r)

print(f"Wrote CSV: {CSV_PATH} (rows={len(rows)})")

Wrote CSV: ../04-output/raw/biostars/biostars_metagenomics.csv (rows=0)
